# Task 3 – Final outputs (run 3, full-precision inference, JPEG q75)

Findings from `task3_output_format_study.ipynb`:
* The training notebook saved its outputs as **JPEG q95**. Re-saving the same generator's outputs at **q75** (the
  dataset's own quality) improves FID by ~1.2 (102.10 → 100.90); regenerating at q95 gives 102.25, so the gain comes
  from the JPEG quality setting, not from fp32 vs bf16.
* **JPEG q75** matches the real dataset exactly (the `real-tables` row is identical to q75), and is better than PNG
  (lossless outputs look *less* like the JPEG-compressed real images to Inception). q65 was 0.04 better – within noise,
  so we keep the principled choice: the same encoding as the dataset.

This notebook re-generates **all** `pred_B2A` (7038) and `pred_A2B` (300) from the run-3 best checkpoint in fp32,
saves them as JPEG q75 into `outputs/`, and writes `submission.csv` with the class evaluation code.
The previous q95 outputs are kept in `outputs_run3/`.

In [1]:
import glob, json, os, re, time
from pathlib import Path
import numpy as np, pandas as pd, scipy.linalg, torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models, torchvision.transforms as T
from PIL import Image
from scipy.spatial.distance import cosine

HERE = Path.cwd() if (Path.cwd() / "outputs").exists() else Path.cwd().parent
os.chdir(HERE)
DATA = HERE.parent / "data"
CKPT = "checkpoints/20261004_065303_generators_best.pt"
NB = "src/task3_run3.ipynb"
dev = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
assert (HERE / "outputs_run3/pred_B2A").exists(), "back up the run-3 outputs to outputs_run3/ first"
print("working dir:", HERE, "| device:", dev)

working dir: /Users/manjotkaur/Desktop/DATA 266/lab1/task3_gan/manjot | device: mps


## 1. Rebuild the run-3 generators

In [2]:
ns = {"torch": torch, "nn": nn, "F": F, "np": np}
cells = ["".join(c["source"]) for c in json.load(open(NB))["cells"] if c["cell_type"] == "code"]
for src in cells:
    if "class Generator" in src or "class ResBlock" in src or "class ResidualBlock" in src:
        try:
            exec(src, ns)
        except Exception as e:   # cell also builds models with `device`; class definitions above that line are kept
            print("  (note: model cell partly executed:", type(e).__name__, e, ")")
for src in cells:
    m = re.search(r"^G_CFG\s*=\s*dict\((.*?)\)\s*$", src, re.S | re.M)
    if m and "G_CFG" not in ns:
        exec("G_CFG = dict(" + m.group(1) + ")", ns)
ck = torch.load(CKPT, map_location="cpu")
cfg = ck.get("G_CFG", ns.get("G_CFG", {}))
print("G_CFG:", cfg)
G = {}
for k in ("G_AB", "G_BA"):
    G[k] = ns["Generator"](**cfg)
    G[k].load_state_dict(ck[k]); G[k].to(dev).eval().float()

  (note: model cell partly executed: NameError name 'device' is not defined )
G_CFG: {'ngf': 64, 'n_res': 9, 'upsample': 'convtranspose'}


## 2. Generate everything in fp32 and save as JPEG q75

In [3]:
to_in = T.Compose([T.ToTensor(), T.Normalize((0.5,) * 3, (0.5,) * 3)])

@torch.no_grad()
def run(g, src_paths, out_dir, bs=16):
    out_dir.mkdir(parents=True, exist_ok=True)
    for f in out_dir.glob("*"):
        f.unlink()                                   # clear old (bf16) files
    t0 = time.time()
    for i in range(0, len(src_paths), bs):
        chunk = src_paths[i:i + bs]
        x = torch.stack([to_in(Image.open(p).convert("RGB")) for p in chunk]).to(dev)
        y = (g(x).float().clamp(-1, 1) * 0.5 + 0.5).cpu()
        for p, img in zip(chunk, y):
            arr = (img.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)
            Image.fromarray(arr).save(out_dir / (Path(p).stem + ".jpg"), quality=75)
    print(f"{out_dir.name}: {len(src_paths)} images in {time.time() - t0:.0f}s")

photos = sorted(glob.glob(str(DATA / "photo_jpg/*.jpg")))
monets = sorted(glob.glob(str(DATA / "monet_jpg/*.jpg")))
run(G["G_BA"], photos, HERE / "outputs/pred_B2A")      # Photo -> Monet (7038)
run(G["G_AB"], monets, HERE / "outputs/pred_A2B")      # Monet -> Photo (300)
print(len(os.listdir("outputs/pred_B2A")), len(os.listdir("outputs/pred_A2B")))

pred_B2A: 7038 images in 374s


pred_A2B: 300 images in 16s
7038 300


## 3. Class evaluation script → `submission.csv`

In [4]:
inc = models.inception_v3(weights=models.Inception_V3_Weights.IMAGENET1K_V1, transform_input=False)
inc.fc = nn.Identity(); inc.to(dev).eval()
TF = T.Compose([T.Resize(299), T.CenterCrop(299), T.ToTensor(),
                T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])

@torch.no_grad()
def acts(paths):
    return np.concatenate([inc(torch.stack([TF(Image.open(p).convert("RGB")) for p in paths[i:i + 32]]).to(dev)).cpu().numpy()
                           for i in range(0, len(paths), 32)])

def fid_mifid(real, gen, n=300):
    r, g = acts(sorted(real)[:n]), acts(sorted(gen)[:n])
    s1, s2 = np.cov(r, rowvar=False), np.cov(g, rowvar=False)
    c = scipy.linalg.sqrtm(s1.dot(s2)).real
    d = r.mean(0) - g.mean(0)
    return float(d.dot(d) + np.trace(s1 + s2 - 2 * c)), float(np.mean([cosine(r[i], g[i]) for i in range(len(r))]))

fid_B2A, mifid_B2A = fid_mifid(monets, glob.glob("outputs/pred_B2A/*.jpg"))
fid_A2B, mifid_A2B = fid_mifid(photos, glob.glob("outputs/pred_A2B/*.jpg"))
print(f"[Photo->Monet] FID={fid_B2A:.3f}  MiFID={mifid_B2A:.4f}")
print(f"[Monet->Photo] FID={fid_A2B:.3f}  MiFID={mifid_A2B:.4f}")
sub = pd.DataFrame([{"ID": 1, "FID": (fid_A2B + fid_B2A) / 2, "MiFID": (mifid_A2B + mifid_B2A) / 2}])
sub.to_csv("outputs/submission.csv", index=False)
print(sub, "\nKaggle score =", round(-(sub.FID[0] + sub.MiFID[0]) / 2, 3))

[Photo->Monet] FID=99.041  MiFID=0.4093
[Monet->Photo] FID=102.750  MiFID=0.4201
   ID         FID     MiFID
0   1  100.895632  0.414712 
Kaggle score = -50.655
